# 02 — Frozen semantic inference

**Inputs:** an external development PLY and compatible Run 1 selected `.pt` checkpoint. Neither is bundled. Install `.[geometry,analysis]`. Set `PLY_PATH`, `CHECKPOINT` and the correct `SAMPLE_ID` in the input cell. CPU is the default.

Pipeline: finite full metric cloud → centroid/unit-sphere model normalization → deterministic 10k sample → frozen XYZ-only PointNet++ → nearest-neighbor full-cloud labels. RGB never enters the model. The shared `analyze_plant` composition also computes downstream proxies; this notebook inspects its semantic outputs, avoiding a second implementation of preprocessing/projection.

CI injects a synthetic semantic backend; that checks execution, not Run 1 model accuracy. Real use follows the external-checkpoint path below. Test plants are out of scope.


In [ ]:
from pathlib import Path
import numpy as np

# Set these public inputs before running all cells. Never supply test plants.
# The optional context is used only by the documented synthetic CI smoke test.
CONTEXT = globals().get("_AGRI3D_NOTEBOOK_CONTEXT", {})
PLY_PATH = CONTEXT.get("ply_path", None)
SHOW_VIEWS = CONTEXT.get("visualize", True)
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise ValueError("Launch this notebook from the checkout or its notebooks directory")
if PLY_PATH is None:
    raise ValueError("Set PLY_PATH to an external development PLY, then run all cells")
PLY_PATH = Path(PLY_PATH)

CHECKPOINT = CONTEXT.get("checkpoint", None)
SAMPLE_ID = CONTEXT.get("sample_id", "example-plant")  # use the manifest ID for reproduction
CONFIG = PROJECT_ROOT / "configs" / "phenotyping_pipeline.yaml"
DEVICE = "cpu"


In [ ]:
from agri3d.inference.pointnet2_backend import PyTorchPointNet2Predictor
from agri3d.pipeline.analyze import analyze_plant
from agri3d.pipeline.config import PipelineConfig

predictor = CONTEXT.get("predictor")  # synthetic smoke-test backend only; normally unset
if predictor is None:
    if CHECKPOINT is None:
        raise ValueError("Set CHECKPOINT to a compatible external Run 1 best.pt")
    predictor = PyTorchPointNet2Predictor(CHECKPOINT, device=DEVICE)
result = analyze_plant(PLY_PATH, predictor=predictor,
                       config=PipelineConfig.load(CONFIG), sample_id=SAMPLE_ID)


In [ ]:
print("Sampled input shape:", result.sampled.xyz.shape)
print("Valid sample slots:", int(result.sampled.attributes["valid_mask"].sum()))
print("Sampled leaf/stalk predictions:", np.bincount(result.prediction.labels, minlength=2))
print("Checkpoint metadata:", result.prediction.model_metadata)


## Full-cloud projection inspection

The shared pipeline has already projected labels onto the cleaned full metric cloud. Original row identity remains in `source_index`. This is inference output, not a biological instance reference or an evaluation claim.


In [ ]:
print("Full cloud / labels:", result.cloud.xyz.shape, result.semantic_labels.shape)
assert len(result.semantic_labels) == len(result.cloud.xyz)
print("Full-cloud leaf/stalk counts:", np.bincount(result.semantic_labels, minlength=2))
